# MoH Chuyên khoa × VM14K — pilot: does grounding an LLM in the official guideline help?

**Câu hỏi pilot:** với một chuyên khoa cụ thể, nếu cho LLM đọc trích đoạn hướng dẫn
chuyên môn của Bộ Y Tế (`data/Chuyên khoa/<tên chuyên khoa>/`) trước khi trả lời câu hỏi
trắc nghiệm VM14K của đúng chuyên khoa đó, độ chính xác có tăng so với hỏi thẳng
(zero-shot, không có tài liệu) hay không?

Thiết kế: với mỗi câu hỏi được lấy mẫu, mỗi LLM được hỏi **2 lần** —
- **baseline**: chỉ câu hỏi + 4 lựa chọn, không có ngữ cảnh.
- **rag**: câu hỏi + 4 lựa chọn + top-k đoạn trích liên quan nhất (TF-IDF) từ tài liệu
  MoH của đúng chuyên khoa.

So sánh accuracy hai điều kiện, theo từng LLM. Đây là **pilot nhỏ để quyết định có đáng
đầu tư tiếp hay không**, không phải một benchmark đã được kiểm chứng — xem phần "Giới
hạn" ở cuối notebook.

> Nền tảng dữ liệu: `docs/cleaning/CLEANING.md` (VM14K) và `data/MoH_corpus_spec.md`
> (kho MoH). Notebook này **không** phụ thuộc vào việc kho MoH đã "freeze" theo spec đó —
> nó tự trích văn bản on-the-fly cho một chuyên khoa duy nhất, không đụng tới toàn bộ 112
> file.

## Cách chạy

1. **Hôm nay (máy này, không cần internet/API key):** `Runtime → Run all` chạy được hết
   tới hết phần trích PDF + lấy mẫu câu hỏi + xây retrieval. Phần gọi LLM sẽ tự bỏ qua
   (in cảnh báo) nếu thiếu package hoặc thiếu API key — không crash notebook.
2. **Ngày mai (trên VM, sau khi thầy connect lại):**
   - Cài đặt (nếu thiếu): `pip install anthropic openai pymupdf pdfplumber`.
   - Set biến môi trường `ANTHROPIC_API_KEY` và/hoặc `OPENAI_API_KEY`.
   - Sửa `RUN_LLM_CALLS = True` ở cell config, `Run all` lại.
3. Đổi chuyên khoa/model/số câu mẫu chỉ bằng cách sửa các biến trong **cell config** —
   không cần sửa gì bên dưới.

In [1]:
# 1. Imports và cấu hình
import hashlib
import json
import os
import random
import re
import subprocess
import sys
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- pilot config — sửa các dòng này để đổi chuyên khoa / model / cỡ mẫu -------------

# Tên chuyên khoa CHUẨN (đã sửa mojibake, xem cell 4) — phải khớp một trong các key in ra
# ở cell "5. Bảng ánh xạ chuyên khoa". Mặc định chọn Tim mạch: mapping 1-1 rõ ràng sang
# medical_topic="Cardiology", 262 câu VM14K, kho MoH gọn (6 file, ~1 file thiếu part01 —
# xem cảnh báo lúc trích).
SPECIALTY_VN = "Tim mạch"

# 1-2 LLM lớn để so sánh. key tuỳ đặt tên, "provider" phải là "anthropic" hoặc "openai".
# Model id là chỗ NHIỀU KHẢ NĂNG cần sửa lại cho khớp quyền truy cập thực tế trên VM.
MODELS = {
    "claude": {"provider": "anthropic", "model": "claude-sonnet-5"},
    "gpt": {"provider": "openai", "model": "gpt-4o"},
}

N_SAMPLE = 20          # số câu VM14K lấy mẫu cho chuyên khoa này
TOP_K_CHUNKS = 3        # số đoạn trích ngữ cảnh cho điều kiện "rag"
CHUNK_WORDS = 180        # kích thước 1 chunk (từ)
CHUNK_OVERLAP = 40        # số từ chồng lấp giữa 2 chunk liên tiếp

# An toàn: mặc định KHÔNG gọi API (để chạy full notebook hôm nay không tốn tiền / không
# cần internet). Đổi thành True trên VM sau khi đã set API key.
RUN_LLM_CALLS = False

print(f"SPECIALTY_VN = {SPECIALTY_VN!r}")
print(f"MODELS = {MODELS}")
print(f"RUN_LLM_CALLS = {RUN_LLM_CALLS}")

SPECIALTY_VN = 'Tim mạch'
MODELS = {'claude': {'provider': 'anthropic', 'model': 'claude-sonnet-5'}, 'gpt': {'provider': 'openai', 'model': 'gpt-4o'}}
RUN_LLM_CALLS = False


In [2]:
# 2. Tự tìm repo (local checkout, hoặc git clone nếu chạy notebook lẻ trên máy khác)
REPO_URL = "https://github.com/hanguyennn2812/VM14K_Research.git"
REPO_BRANCH = "main"
CLONE_ROOT = Path("./_VM14K_Research_clone")

candidate_roots = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = None
for cand in candidate_roots:
    if (cand / "data" / "cleaned" / "clean_final.jsonl").exists():
        REPO_ROOT = cand
        break

if REPO_ROOT is None:
    if CLONE_ROOT.exists():
        print(f"Dùng clone có sẵn: {CLONE_ROOT.resolve()}")
    else:
        print(f"Không tìm thấy checkout cục bộ — git clone (branch={REPO_BRANCH!r}) ...")
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(CLONE_ROOT)],
            check=True,
        )
    REPO_ROOT = CLONE_ROOT

DATA_PATH = REPO_ROOT / "data" / "cleaned" / "clean_final.jsonl"
CK_PARENT = REPO_ROOT / "data"
CACHE_DIR = REPO_ROOT / "data" / "interim" / "moh_extract_cache"
REPORT_DIR = REPO_ROOT / "reports" / "analysis"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"Không thấy {DATA_PATH} — kiểm tra lại checkout"
print(f"REPO_ROOT = {REPO_ROOT.resolve()}")
print(f"DATA_PATH = {DATA_PATH}")

REPO_ROOT = F:\VM14K_Research
DATA_PATH = F:\VM14K_Research\data\cleaned\clean_final.jsonl


## 3. Thư mục `Chuyên khoa` — lưu ý encoding

File/thư mục trong `data/Chuyên khoa/` được giải nén với **tên bị mojibake**: byte UTF-8
gốc bị đọc nhầm bằng bảng mã CP437/CP850 rồi ghi lại, nên tên thư mục thật trên đĩa trông
như `"Chuy├¬n khoa"` chứ không phải `"Chuyên khoa"`. Đây là lỗi encoding lúc giải nén, có
thật trên đĩa (không phải lỗi hiển thị terminal) — mọi tool (Python, git, shell) đều thấy
cùng chuỗi mojibake này.

Thay vì gõ tay chuỗi mojibake (dễ sai 1 byte là hỏng path), notebook tự đảo ngược phép biến
đổi (`encode('cp437').decode('utf-8')`) để lấy lại tên tiếng Việt đúng, dùng để khớp với
bảng ánh xạ chuyên khoa bên dưới — đã kiểm tra khớp with cả 27 thư mục.

In [3]:
# 4. Liệt kê thư mục chuyên khoa thật trên đĩa + sửa mojibake để lấy tên hiển thị đúng
def fix_mojibake(name: str) -> str:
    """UTF-8 bytes bị đọc nhầm bằng CP437 lúc giải nén -> đảo ngược lại. No-op nếu fail."""
    try:
        return name.encode("cp437").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return name


_ck_candidates = [p for p in CK_PARENT.iterdir() if p.is_dir() and fix_mojibake(p.name) == "Chuyên khoa"]
assert _ck_candidates, "Không tìm thấy thư mục 'Chuyên khoa' (kể cả sau khi sửa mojibake) dưới data/"
CK_DIR = _ck_candidates[0]

specialty_dirs = sorted(
    (fix_mojibake(p.name), p) for p in CK_DIR.iterdir() if p.is_dir()
)
print(f"Số thư mục chuyên khoa trên đĩa: {len(specialty_dirs)}")
for display_name, _ in specialty_dirs:
    print(" -", display_name)

# data/MoH_corpus_spec.md nói "26 specialty folders" (khớp với 26 section trong
# van_ban_theo_chuyen_khoa.tex). Trên đĩa thực tế có 27 — "Trạm y tế" không nằm trong
# mục lục .tex. Đây là một discrepancy thật (giống kiểu 112-vs-109 file đã nêu trong
# spec), không tự ý bỏ qua — chỉ cảnh báo.
if len(specialty_dirs) != 26:
    print(
        f"\n[CẢNH BÁO] {len(specialty_dirs)} thư mục trên đĩa, nhưng "
        "data/MoH_corpus_spec.md ghi nhận 26 (khớp mục lục van_ban_theo_chuyen_khoa.tex). "
        "Thư mục lệch: 'Trạm y tế' (Y tế cấp xã) không có trong mục lục .tex — có thể là "
        "thư mục bổ sung sau khi .tex được tạo. Cần đối chiếu lại trước khi freeze corpus, "
        "không liên quan tới pilot này."
    )

Số thư mục chuyên khoa trên đĩa: 27
 - Cấp cứu - Sơ cứu
 - Da liễu
 - Dinh dưỡng
 - Dược học - Dược lâm sàng
 - Gây mê hồi sức
 - HIV
 - Huyết học - Truyền máu
 - Hô hấp
 - Ký sinh trùng
 - Nhi khoa
 - Nhiễm khuẩn học - Vi sinh
 - Nhãn khoa
 - Nội tiết - Đái tháo đường
 - Pháp y
 - Phục hồi chức năng
 - STDs
 - Sản phụ khoa
 - Thần kinh
 - Thận học
 - Tim mạch
 - Tiết niệu
 - Truyền nhiễm
 - Trạm y tế
 - Tâm thần - Nghiện chất
 - Ung bướu
 - Y học cổ truyền
 - Y tế công cộng - Dự phòng

[CẢNH BÁO] 27 thư mục trên đĩa, nhưng data/MoH_corpus_spec.md ghi nhận 26 (khớp mục lục van_ban_theo_chuyen_khoa.tex). Thư mục lệch: 'Trạm y tế' (Y tế cấp xã) không có trong mục lục .tex — có thể là thư mục bổ sung sau khi .tex được tạo. Cần đối chiếu lại trước khi freeze corpus, không liên quan tới pilot này.


## 5. Bảng ánh xạ chuyên khoa (tiếng Việt) → `medical_topic` (VM14K, tiếng Anh)

VM14K gắn nhãn câu hỏi bằng phân loại 34 chuyên khoa tiếng Anh của paper (xem
`scripts/analysis/propose_topic_mapping.py`). Kho MoH đặt tên chuyên khoa tiếng Việt theo
cách khác. Hai hệ thống không khớp 1-1 hoàn toàn — bảng dưới đây là **đề xuất ánh xạ thủ
công**, cùng tinh thần AUTO/AMBIGUOUS/JUNK của `propose_topic_mapping.py`:

- `confidence="clean"` — ánh xạ rõ ràng, dùng được ngay cho pilot.
- `confidence="ambiguous"` — có target gần nhất nhưng còn tranh cãi (ví dụ "Pháp y" vs
  "Pathology" — pháp y rộng hơn pathology).
- `confidence="gap"` — không có target 1-1 hợp lý trong 34 chuyên khoa của VM14K
  (ví dụ "Dinh dưỡng", "Dược học lâm sàng" — những mảng này trải khắp nhiều chuyên khoa
  VM14K chứ không dồn vào một nhãn). Chọn các chuyên khoa `gap` sẽ khiến bước lọc câu hỏi
  phía dưới báo lỗi rõ ràng thay vì âm thầm trả về tập rỗng.

In [4]:
# 6. Bảng ánh xạ Chuyên khoa (VN, đã sửa mojibake) -> medical_topic (EN, VM14K)
# format: "tên thư mục VN": (["medical_topic", ...], "clean" | "ambiguous" | "gap", "ghi chú")
SPECIALTY_MAP: dict[str, tuple[list[str], str, str]] = {
    "Cấp cứu - Sơ cứu":            (["Emergency Medicine"], "clean", ""),
    "Da liễu":                      (["Dermatology"], "clean", ""),
    "Dinh dưỡng":                    ([], "gap", "Nutrition trải khắp Internal Medicine/Endocrinology/Public Health"),
    "Dược học - Dược lâm sàng":       ([], "gap", "Pharmacology/dược lâm sàng không có nhãn riêng trong 34 chuyên khoa VM14K"),
    "Gây mê hồi sức":                (["Anesthesiology"], "clean", ""),
    "HIV":                            (["Infectious Diseases"], "clean", ""),
    "Huyết học - Truyền máu":         (["Hematology"], "clean", ""),
    "Hô hấp":                         (["Pulmonology"], "clean", ""),
    "Ký sinh trùng":                   ([], "gap", "Parasitology chồng lấn Infectious Diseases nhưng không ép 1-1"),
    "Nhi khoa":                       (["Pediatrics"], "clean", ""),
    "Nhiễm khuẩn học - Vi sinh":       ([], "gap", "Microbiology chồng lấn Infectious Diseases nhưng không ép 1-1"),
    "Nhãn khoa":                      (["Ophthalmology"], "clean", ""),
    "Nội tiết - Đái tháo đường":      (["Endocrinology"], "clean", ""),
    "Phục hồi chức năng":              (["Physical Medicine and Rehabilitation"], "clean", ""),
    "Pháp y":                         (["Pathology"], "ambiguous", "pháp y rộng hơn pathology; best-guess của propose_topic_mapping.py"),
    "STDs":                           (["Infectious Diseases"], "ambiguous", "cũng liên quan Dermatology/Urology/OB-GYN tuỳ bệnh"),
    "Sản phụ khoa":                   (["Obstetrics and Gynecology"], "clean", ""),
    "Thận học":                       (["Nephrology"], "clean", ""),
    "Thần kinh":                      (["Neurology"], "clean", ""),
    "Tim mạch":                       (["Cardiology"], "clean", ""),
    "Tiết niệu":                      (["Urology"], "clean", ""),
    "Truyền nhiễm":                   (["Infectious Diseases"], "clean", ""),
    "Trạm y tế":                      ([], "gap", "y tế cấp xã / chăm sóc ban đầu — không có nhãn tương ứng"),
    "Tâm thần - Nghiện chất":          (["Psychiatry"], "clean", "\"Nghiện chất\" gộp vào Psychiatry theo propose_topic_mapping.py"),
    "Ung bướu":                       (["Oncology"], "clean", ""),
    "Y học cổ truyền":                (["Eastern Medicine"], "clean", ""),
    "Y tế công cộng - Dự phòng":       (["Public Health", "Preventive Healthcare"], "clean", "2 nhãn VM14K tương ứng 2 nửa tên chuyên khoa"),
}

_map_df = pd.DataFrame(
    [
        {"chuyên khoa (VN)": k, "medical_topic (EN)": ", ".join(v[0]) or "—", "confidence": v[1], "ghi chú": v[2]}
        for k, v in SPECIALTY_MAP.items()
    ]
).sort_values(["confidence", "chuyên khoa (VN)"])
display(_map_df)

_disk_names = {d for d, _ in specialty_dirs}
_missing_on_disk = set(SPECIALTY_MAP) - _disk_names
_missing_in_map = _disk_names - set(SPECIALTY_MAP)
if _missing_on_disk:
    print(f"[CẢNH BÁO] có trong bảng ánh xạ nhưng KHÔNG thấy thư mục: {_missing_on_disk}")
if _missing_in_map:
    print(f"[CẢNH BÁO] có thư mục trên đĩa nhưng chưa có trong bảng ánh xạ: {_missing_in_map}")

,chuyên khoa (VN),medical_topic (EN),confidence,ghi chú
14,Pháp y,Pathology,ambiguous,pháp y rộng hơn pathology; best-guess của prop...
15,STDs,Infectious Diseases,ambiguous,cũng liên quan Dermatology/Urology/OB-GYN tuỳ ...
0,Cấp cứu - Sơ cứu,Emergency Medicine,clean,
1,Da liễu,Dermatology,clean,
4,Gây mê hồi sức,Anesthesiology,clean,
5,HIV,Infectious Diseases,clean,
6,Huyết học - Truyền máu,Hematology,clean,
7,Hô hấp,Pulmonology,clean,
9,Nhi khoa,Pediatrics,clean,
11,Nhãn khoa,Ophthalmology,clean,


In [5]:
# 7. Chốt chuyên khoa cho lần chạy này (theo SPECIALTY_VN ở cell config)
assert SPECIALTY_VN in SPECIALTY_MAP, (
    f"{SPECIALTY_VN!r} không có trong SPECIALTY_MAP. Chọn 1 trong: {sorted(SPECIALTY_MAP)}"
)
canonical_topics, confidence, note = SPECIALTY_MAP[SPECIALTY_VN]
assert confidence != "gap" and canonical_topics, (
    f"{SPECIALTY_VN!r} là mapping 'gap' (không có medical_topic tương ứng rõ ràng: {note}). "
    "Chọn một chuyên khoa 'clean' hoặc 'ambiguous' khác trong SPECIALTY_MAP."
)
if confidence == "ambiguous":
    print(f"[LƯU Ý] mapping 'ambiguous' cho {SPECIALTY_VN!r}: {note}")

specialty_dir = dict(specialty_dirs)[SPECIALTY_VN]
print(f"SPECIALTY_VN    = {SPECIALTY_VN!r}")
print(f"canonical_topics = {canonical_topics}")
print(f"specialty_dir    = {specialty_dir}")

SPECIALTY_VN    = 'Tim mạch'
canonical_topics = ['Cardiology']
specialty_dir    = F:\VM14K_Research\data\Chuy├¬n khoa\Tim mß║ích


## 8. Trích văn bản PDF cho chuyên khoa đã chọn

Theo `data/MoH_corpus_spec.md` §2–§3:
- Chỉ dùng file **born-digital** (trích được text sạch). File **scanned** (< 50 ký tự/trang)
  bị loại, có log lý do — không âm thầm bỏ qua (§4).
- File `..._partNN.pdf` là các phần của **cùng một** văn bản gốc — nối lại theo thứ tự
  trước khi coi là 1 tài liệu (§3.2). Nếu thiếu 1 phần (ví dụ chỉ thấy `part02`), vẫn dùng
  phần có sẵn nhưng cảnh báo rõ — pilot này KHÔNG cần đợi tới khi kho MoH "freeze" đầy đủ.

Dùng PyMuPDF (`fitz`) để trích — nhanh và giữ đúng dấu tiếng Việt; `pdfplumber` làm phương
án dự phòng nếu `fitz` lỗi trên 1 file cụ thể. Kết quả được cache ra
`data/interim/moh_extract_cache/` (không commit — xem `.gitignore`) để chạy lại không phải
trích lại PDF.

In [6]:
# 9. Hàm trích PDF (born-digital only) + gộp multi-part + cache
SCAN_CHARS_PER_PAGE_FLOOR = 50  # ngưỡng đề xuất trong MoH_corpus_spec.md §4


def _extract_with_fitz(path: Path) -> tuple[str, int]:
    import fitz
    doc = fitz.open(path)
    text = "\n".join(page.get_text() for page in doc)
    return text, doc.page_count


def _extract_with_pdfplumber(path: Path) -> tuple[str, int]:
    import pdfplumber
    with pdfplumber.open(path) as pdf:
        text = "\n".join(page.extract_text() or "" for page in pdf.pages)
        return text, len(pdf.pages)


def extract_pdf_text(path: Path) -> tuple[str, int]:
    """(text, n_pages). Thử fitz trước, pdfplumber làm dự phòng."""
    try:
        return _extract_with_fitz(path)
    except Exception as exc_fitz:
        try:
            return _extract_with_pdfplumber(path)
        except Exception as exc_plumber:
            raise RuntimeError(f"Trích {path.name} thất bại: fitz={exc_fitz!r} pdfplumber={exc_plumber!r}")


_PART_RE = re.compile(r"^(?P<base>.+?)_part(?P<num>\d+)(?P<suffix>\s*\(\d+\))?\.pdf$", re.IGNORECASE)


def group_parts(pdf_paths: list[Path]) -> dict[str, list[Path]]:
    """Gộp các file '<ten>_partNN.pdf' về cùng 1 doc_id; file thường -> doc_id riêng nó."""
    groups: dict[str, list[tuple[int, Path]]] = {}
    for p in pdf_paths:
        m = _PART_RE.match(p.name)
        if m:
            key = m.group("base")
            groups.setdefault(key, []).append((int(m.group("num")), p))
        else:
            groups.setdefault(p.stem, []).append((0, p))
    return {k: [p for _, p in sorted(v)] for k, v in groups.items()}


def extract_specialty_documents(folder: Path, cache_path: Path, force: bool = False) -> list[dict]:
    """Trích toàn bộ PDF born-digital trong 1 thư mục chuyên khoa, gộp theo doc, cache JSONL."""
    if cache_path.exists() and not force:
        with cache_path.open(encoding="utf-8") as fh:
            return [json.loads(line) for line in fh if line.strip()]

    pdf_paths = sorted(folder.glob("*.pdf"))
    groups = group_parts(pdf_paths)
    docs = []
    for doc_id, parts in sorted(groups.items()):
        if len(parts) > 1:
            expected = set(range(1, len(parts) + 1))
            found = set()
            for p in parts:
                m = _PART_RE.match(p.name)
                if m:
                    found.add(int(m.group("num")))
            missing = expected - found
            if missing or min(found, default=1) != 1:
                print(f"  [CẢNH BÁO] '{doc_id}': part nghi thiếu — file có part {sorted(found)}, nối lại đúng những gì có.")

        texts, total_pages, sources = [], 0, []
        for p in parts:
            try:
                text, n_pages = extract_pdf_text(p)
            except RuntimeError as exc:
                print(f"  [LỖI] {exc}")
                continue
            texts.append(text)
            total_pages += n_pages
            sources.append(p.name)

        full_text = "\n\n".join(texts)
        non_ws_chars = len(re.sub(r"\s+", "", full_text))
        chars_per_page = non_ws_chars / total_pages if total_pages else 0.0
        doc_class = "scanned_dropped" if chars_per_page < SCAN_CHARS_PER_PAGE_FLOOR else "born_digital"

        docs.append({
            "doc_id": doc_id,
            "source_files": sources,
            "pages": total_pages,
            "chars": non_ws_chars,
            "chars_per_page": round(chars_per_page, 1),
            "class": doc_class,
            "text": full_text if doc_class == "born_digital" else "",
        })

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("w", encoding="utf-8") as fh:
        for d in docs:
            fh.write(json.dumps(d, ensure_ascii=False) + "\n")
    return docs

In [7]:
# 10. Chạy trích cho chuyên khoa đã chọn (cache theo tên chuyên khoa)
def slugify(name: str) -> str:
    norm = unicodedata.normalize("NFKD", name)
    ascii_only = norm.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-zA-Z0-9]+", "_", ascii_only).strip("_").lower()


specialty_slug = slugify(SPECIALTY_VN)
cache_path = CACHE_DIR / f"{specialty_slug}.jsonl"

t0 = time.time()
documents = extract_specialty_documents(specialty_dir, cache_path)
print(f"Trích {len(documents)} tài liệu ({time.time() - t0:.1f}s), cache: {cache_path}")

usable_docs = [d for d in documents if d["class"] == "born_digital"]
dropped_docs = [d for d in documents if d["class"] != "born_digital"]
print(f"born_digital: {len(usable_docs)} | scanned_dropped: {len(dropped_docs)}")

if not usable_docs:
    print(
        "[CẢNH BÁO] 0 tài liệu born-digital cho chuyên khoa này — đây đúng là "
        "\"coverage gap\" mà MoH_corpus_spec.md §4.3 dặn phải báo cho supervisor, "
        "không tự OCR để né. Đổi SPECIALTY_VN sang chuyên khoa khác cho pilot."
    )

Trích 6 tài liệu (35.1s), cache: F:\VM14K_Research\data\interim\moh_extract_cache\tim_mach.jsonl
born_digital: 6 | scanned_dropped: 0


In [8]:
# 11. Báo cáo trích xuất theo tài liệu
_report_rows = [
    {
        "doc_id": d["doc_id"][:70],
        "pages": d["pages"],
        "chars": d["chars"],
        "chars/page": d["chars_per_page"],
        "class": d["class"],
        "n_parts": len(d["source_files"]),
    }
    for d in documents
]
display(pd.DataFrame(_report_rows))

,doc_id,pages,chars,chars/page,class,n_parts
0,Quyet_dinh_so_1762_QD_BYT_ngay_17_04_2020_cua_...,22,23001,1045.5,born_digital,1
1,Quyet_dinh_so_2248_QD_BYT_ngay_20_05_2023_V_v_...,38,48219,1268.9,born_digital,1
2,Quyet_dinh_so_2475_QD_BYT_ngay_09_9_2022_cua_B...,47,63953,1360.7,born_digital,1
3,Quyet_dinh_so_3908_QD_BYT_ngay_20_10_2023_cua_...,81,120237,1484.4,born_digital,1
4,Quyet_dinh_so_5332_QD_BYT_ngay_23_12_2020_cua_...,172,216164,1256.8,born_digital,1
5,Quyet_dinh_so_5333_QD_BYT_ngay_23_12_2020_cua_...,128,237594,1856.2,born_digital,1


## 12. Retrieval đơn giản — TF-IDF trên chunk

Mục tiêu là pilot nhanh, chạy offline được (không cần tải embedding model), nên dùng
`TfidfVectorizer` (đã có sẵn trong `requirements.txt`) thay vì vector DB/embedding API.
Đủ để lấy top-k đoạn liên quan nhất cho một câu hỏi trắc nghiệm ngắn.

In [9]:
# 13. Chunk hoá + TF-IDF index
def chunk_text(text: str, chunk_words: int, overlap: int) -> list[str]:
    words = text.split()
    if not words:
        return []
    step = max(chunk_words - overlap, 1)
    return [
        " ".join(words[i : i + chunk_words])
        for i in range(0, len(words), step)
        if words[i : i + chunk_words]
    ]


chunks: list[str] = []
chunk_doc_ids: list[str] = []
for d in usable_docs:
    doc_chunks = chunk_text(d["text"], CHUNK_WORDS, CHUNK_OVERLAP)
    chunks.extend(doc_chunks)
    chunk_doc_ids.extend([d["doc_id"]] * len(doc_chunks))

print(f"Tổng số chunk: {len(chunks)} (từ {len(usable_docs)} tài liệu)")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

_vectorizer = TfidfVectorizer(max_features=20000)
_chunk_matrix = _vectorizer.fit_transform(chunks) if chunks else None


def retrieve_context(query: str, k: int = TOP_K_CHUNKS) -> list[str]:
    if _chunk_matrix is None:
        return []
    q_vec = _vectorizer.transform([query])
    sims = cosine_similarity(q_vec, _chunk_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    return [chunks[i] for i in top_idx if sims[i] > 0]

Tổng số chunk: 1399 (từ 6 tài liệu)


## 14. Lấy mẫu câu hỏi VM14K theo chuyên khoa

In [10]:
# 15. Đọc clean_final.jsonl, lọc theo canonical_topics, lấy mẫu N_SAMPLE
records = []
with DATA_PATH.open(encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            records.append(json.loads(line))
df = pd.DataFrame(records)

topic_set = set(canonical_topics)
mask = df["medical_topic"].map(lambda topics: bool(topic_set.intersection(topics or [])))
subset = df[mask]
print(f"Số câu VM14K khớp {canonical_topics}: {len(subset)}")

assert len(subset) > 0, f"0 câu VM14K khớp {canonical_topics} — kiểm tra lại SPECIALTY_MAP"

n = min(N_SAMPLE, len(subset))
sample_df = subset.sample(n=n, random_state=SEED).reset_index(drop=True)
print(f"Lấy mẫu {n} câu (SEED={SEED})")
display(sample_df[["id", "difficulty_level", "medical_topic", "question", "answer"]].head())

Số câu VM14K khớp ['Cardiology']: 262
Lấy mẫu 20 câu (SEED=42)


,id,difficulty_level,medical_topic,question,answer
0,9a2169b0c11d426caba7516c9851b4fd,Medium,"[Cardiology, Nephrology]",Amlodipin được chỉ định trong các trường hợp sau:,D
1,6863201cec32445dab2ee2867f34d132,Medium,"[Endocrinology, Cardiology, Pharmacology, Inte...",Niacin trong điều trị tăng lipid máu chính là ...,A
2,ac48dca8e6ae4b69ad0d10f173eb752d,Medium,"[Pharmacology, Cardiology, Gastroenterology, H...",PHÂN LOẠI theo TÍNH CHẤT của ADR MỞ RỘNG thì P...,C
3,048bfbf5f75340c69a68961243f19ab6,Challenging,"[Cardiology, Rheumatology, Pediatrics]",Yếu tố ảnh hưởng đến sự phát sinh của bệnh thấp,B
4,4d8afc3c46d4428daa1505cab7a6e3a4,Challenging,"[Cardiology, Pharmacology]",Lựa chọn thuốc nào sau đây cho bệnh đau thắt n...,D


## 16. Gọi LLM — chỉ chạy khi `RUN_LLM_CALLS = True` và có API key

Đặt `ANTHROPIC_API_KEY` / `OPENAI_API_KEY` trong biến môi trường trước khi chạy (không
hard-code key vào notebook). Cell dưới cố cài package còn thiếu; nếu không có mạng
(chạy hôm nay, chưa có VM) thì chỉ in cảnh báo, không làm hỏng phần còn lại của notebook.

In [11]:
# 17. Cài package client LLM (best-effort — không internet thì bỏ qua, không crash)
def ensure_package(pip_name: str, import_name: str | None = None) -> bool:
    import importlib
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
        return True
    except ImportError:
        pass
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True, timeout=120)
        importlib.import_module(import_name)
        return True
    except Exception as exc:
        print(f"[CẢNH BÁO] không cài/import được {pip_name!r}: {exc!r} — provider liên quan sẽ bị bỏ qua.")
        return False


HAVE_ANTHROPIC = ensure_package("anthropic")
HAVE_OPENAI = ensure_package("openai")

HAVE_ANTHROPIC_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
HAVE_OPENAI_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print(f"anthropic sdk={HAVE_ANTHROPIC} key_set={HAVE_ANTHROPIC_KEY}")
print(f"openai    sdk={HAVE_OPENAI} key_set={HAVE_OPENAI_KEY}")

anthropic sdk=True key_set=False
openai    sdk=True key_set=False


In [12]:
# 18. Prompt builder + wrapper gọi LLM thống nhất + parser đáp án
def build_prompt(question: str, options: list[str], context_chunks: list[str]) -> tuple[str, str]:
    letters = [chr(65 + i) for i in range(len(options))]
    options_block = "\n".join(f"{L}. {opt}" for L, opt in zip(letters, options))

    system = (
        "Bạn là bác sĩ chuyên khoa đang làm bài kiểm tra trắc nghiệm y khoa tiếng Việt. "
        "Chỉ trả lời đúng 1 chữ cái tương ứng đáp án đúng nhất, theo định dạng "
        "\"Đáp án: X\" — không giải thích thêm."
    )
    context_block = ""
    if context_chunks:
        joined = "\n---\n".join(context_chunks)
        context_block = (
            "Tài liệu chuyên môn tham khảo (Bộ Y Tế):\n"
            f"\"\"\"\n{joined}\n\"\"\"\n\n"
        )
    user = f"{context_block}Câu hỏi: {question}\n{options_block}\n\nĐáp án:"
    return system, user


_ANSWER_RE = re.compile(r"Đáp\s*án\s*[:\-]?\s*([A-G])", re.IGNORECASE)
_FALLBACK_RE = re.compile(r"\b([A-G])\b")


def parse_answer_letter(raw: str) -> str | None:
    m = _ANSWER_RE.search(raw)
    if m:
        return m.group(1).upper()
    m = _FALLBACK_RE.search(raw.strip())
    return m.group(1).upper() if m else None


def call_llm(provider: str, model: str, system: str, user: str) -> str:
    if provider == "anthropic":
        import anthropic
        client = anthropic.Anthropic()
        resp = client.messages.create(
            model=model,
            max_tokens=32,
            system=system,
            messages=[{"role": "user", "content": user}],
        )
        return "".join(block.text for block in resp.content if hasattr(block, "text"))
    if provider == "openai":
        import openai
        client = openai.OpenAI()
        resp = client.chat.completions.create(
            model=model,
            max_tokens=32,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        )
        return resp.choices[0].message.content or ""
    raise ValueError(f"provider không hỗ trợ: {provider}")


def call_llm_safe(provider: str, model: str, system: str, user: str, retries: int = 1) -> tuple[str | None, str | None]:
    """(raw_text, error). Thử lại `retries` lần với backoff ngắn trước khi bỏ cuộc."""
    last_err = None
    for attempt in range(retries + 1):
        try:
            return call_llm(provider, model, system, user), None
        except Exception as exc:
            last_err = repr(exc)
            if attempt < retries:
                time.sleep(2 * (attempt + 1))
    return None, last_err

In [13]:
# 19. Vòng lặp pilot: mỗi câu x mỗi model x 2 điều kiện (baseline, rag)
active_models = {
    key: cfg
    for key, cfg in MODELS.items()
    if (cfg["provider"] == "anthropic" and HAVE_ANTHROPIC and HAVE_ANTHROPIC_KEY)
    or (cfg["provider"] == "openai" and HAVE_OPENAI and HAVE_OPENAI_KEY)
}

results = []
if not RUN_LLM_CALLS:
    print(
        "RUN_LLM_CALLS=False — bỏ qua phần gọi API (đúng như mặc định khi chạy không có "
        "internet/API key). Đặt RUN_LLM_CALLS=True + set API key trên VM rồi Run all lại."
    )
elif not active_models:
    print("RUN_LLM_CALLS=True nhưng không có model nào sẵn sàng (thiếu SDK và/hoặc API key).")
else:
    print(f"Chạy pilot với model: {list(active_models)} trên {len(sample_df)} câu ...")
    for _, row in sample_df.iterrows():
        options = list(row["options"])
        gold_letter = row["answer"]
        contexts = retrieve_context(row["question"])

        for condition, ctx in (("baseline", []), ("rag", contexts)):
            system, user = build_prompt(row["question"], options, ctx)
            for model_key, cfg in active_models.items():
                raw, err = call_llm_safe(cfg["provider"], cfg["model"], system, user)
                pred_letter = parse_answer_letter(raw) if raw else None
                results.append({
                    "question_id": row["id"],
                    "medical_topic": row["medical_topic"],
                    "difficulty_level": row["difficulty_level"],
                    "model": model_key,
                    "condition": condition,
                    "n_context_chunks": len(ctx),
                    "gold_letter": gold_letter,
                    "pred_letter": pred_letter,
                    "correct": (pred_letter == gold_letter) if pred_letter else False,
                    "raw_response": raw,
                    "error": err,
                })
    print(f"Xong — {len(results)} lượt gọi.")

RUN_LLM_CALLS=False — bỏ qua phần gọi API (đúng như mặc định khi chạy không có internet/API key). Đặt RUN_LLM_CALLS=True + set API key trên VM rồi Run all lại.


In [14]:
# 20. Lưu kết quả
results_df = pd.DataFrame(results)
if not results_df.empty:
    out_path = REPORT_DIR / f"moh_llm_pilot_{specialty_slug}.csv"
    results_df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Đã lưu {len(results_df)} dòng -> {out_path}")
    display(results_df.head())
else:
    print("Chưa có kết quả để lưu (RUN_LLM_CALLS=False hoặc không có model sẵn sàng).")

Chưa có kết quả để lưu (RUN_LLM_CALLS=False hoặc không có model sẵn sàng).


In [15]:
# 21. So sánh accuracy: baseline vs rag, theo từng model
if not results_df.empty:
    summary = (
        results_df.groupby(["model", "condition"])["correct"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "accuracy", "count": "n"})
        .round(3)
    )
    display(summary)

    pivot = results_df.pivot_table(index="model", columns="condition", values="correct", aggfunc="mean")
    ax = pivot.plot.bar(figsize=(7, 4), color={"baseline": "#bd6b4d", "rag": "#315b7d"}, rot=0)
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1)
    ax.set_title(f"{SPECIALTY_VN} ({canonical_topics}) — baseline vs RAG, n={len(sample_df)}/model/condition")
else:
    print("Chưa có kết quả — chạy trên VM với RUN_LLM_CALLS=True trước.")

Chưa có kết quả — chạy trên VM với RUN_LLM_CALLS=True trước.


## Giới hạn của pilot này — đọc trước khi kết luận bất cứ điều gì

- **N nhỏ.** `N_SAMPLE` mặc định 20 câu/chuyên khoa — đủ để sanity-check "có đáng làm tiếp
  không", **không** đủ để kết luận thống kê chắc chắn RAG có giúp ích hay không.
- **Retrieval rất đơn giản** (TF-IDF, không rerank) — nếu RAG không giúp ích, có thể do
  retrieval kém chứ chưa chắc do việc "grounding vào guideline" vô ích.
- **Parser đáp án dựa trên regex** — nếu model trả lời dài dòng bất thường, `pred_letter`
  có thể `None`/sai; kiểm tra cột `raw_response` khi accuracy có vẻ bất thường thấp.
- **Không kiểm chứng y khoa.** Đây là so sánh accuracy so với đáp án gốc VM14K, không phải
  đánh giá bởi chuyên gia — nhất quán với lưu ý "không phải hệ thống tư vấn/chẩn đoán y
  khoa" của `notebooks/VM14K_EDA_Simple_Baseline_Colab.ipynb` và §6 của
  `data/MoH_corpus_spec.md` (nếu pilot này cho thấy đáng làm tiếp theo hướng sinh QA từ
  guideline, hướng đó cần verify của bác sĩ, không phải chạy tự động).
- **Chi phí API.** `len(sample_df) × len(active_models) × 2 điều kiện` lượt gọi mỗi lần
  chạy — kiểm tra `N_SAMPLE` trước khi chạy full trên nhiều chuyên khoa.

**Bước tiếp theo nếu pilot cho tín hiệu tốt:** mở rộng `N_SAMPLE`, chạy nhiều chuyên khoa
`clean` khác (đổi `SPECIALTY_VN`), và cân nhắc retrieval tốt hơn (embedding thay TF-IDF)
trước khi coi đây là kết luận cuối.